# 02 | Exploratory structure and visual diagnostics
**Purpose.** Reproduce the manuscript's descriptive stellar diagrams using the **balanced** FLAME-complete cohort. These plots illustrate catalogue structure; they do not constitute independent spectroscopic validation.

**Input:** `outputs/preprocessed.joblib` from notebook 01.  
**Output:** publication-oriented plots in `outputs/figures/`. The t-SNE calculation may be slow and is controlled by `run_tsne` in the configuration.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if not (ROOT / "config" / "experiment.json").exists():
    ROOT = ROOT.parent
if not (ROOT / "config" / "experiment.json").exists():
    raise FileNotFoundError("Start Jupyter from the repository root or notebooks/ folder.")
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
from IPython.display import display
from src.gaia_pipeline import *
cfg = load_config(ROOT)
np.random.seed(cfg["random_state"])
print("Repository:", ROOT)
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
import inspect
from astropy.coordinates import SkyCoord
import astropy.units as u
from matplotlib.colors import LogNorm
balanced = read_object(ROOT, "preprocessed.joblib")["balanced"]
FIG = output_dir(ROOT) / "figures"
FIG.mkdir(exist_ok=True)
COLORS = {"O": "blue", "B": "lightskyblue", "A": "cyan", "F": "khaki", "G": "yellow", "K": "orange", "M": "red"}
plt.rcParams.update({"figure.dpi": 160, "savefig.dpi": 300})

## Physical and photometric H–R diagrams
The physical diagram uses `Teff` and `Lum-Flame`; the photometric diagram uses corrected G magnitude versus intrinsic BP−RP colour. Filter nonphysical nonpositive values before logarithmic plotting.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for cls, color in COLORS.items():
    subset = balanced[(balanced["SpType-ELS"] == cls) & (balanced["Teff"] > 0) & (balanced["Lum-Flame"] > 0)]
    ax.scatter(subset["Teff"], subset["Lum-Flame"], s=1, alpha=0.5, color=color, label=cls)
ax.set(xscale="log", yscale="log", xlabel="Effective temperature (K)", ylabel="Luminosity ($L/L_\odot$)", title="H–R diagram: temperature and luminosity")
ax.invert_xaxis(); ax.legend(title="Class", markerscale=4); fig.tight_layout()
fig.savefig(FIG / "hr_physical.png"); plt.show()
fig, ax = plt.subplots(figsize=(7, 5))
for cls, color in COLORS.items():
    subset = balanced[balanced["SpType-ELS"] == cls]
    ax.scatter(subset["BP_RP_abs"], subset["G_abs"], s=1, alpha=0.5, color=color, label=cls)
ax.set(xlabel="Corrected BP−RP", ylabel="Absolute G magnitude", title="Photometric H–R diagram")
ax.invert_yaxis(); ax.legend(title="Class", markerscale=4); fig.tight_layout()
fig.savefig(FIG / "hr_photometric.png"); plt.show()

## Equatorial and Galactic density maps
Gaia positions are converted to Galactic coordinates using Astropy. The colour scale is $\log_{10}(\mathrm{count}+1)$.

In [ ]:
coords = SkyCoord(ra=balanced["RA_ICRS"].to_numpy() * u.deg, dec=balanced["DE_ICRS"].to_numpy() * u.deg)
for name, x, y, xlab, ylab in [
    ("galactic", coords.galactic.l.deg, coords.galactic.b.deg, "Galactic longitude (deg)", "Galactic latitude (deg)"),
    ("equatorial", coords.ra.deg, coords.dec.deg, "Right ascension (deg)", "Declination (deg)")
]:
    hist, xe, ye = np.histogram2d(x, y, bins=200)
    fig, ax = plt.subplots(figsize=(8, 4))
    im = ax.imshow(np.log10(hist.T + 1), extent=[xe[0], xe[-1], ye[0], ye[-1]], origin="lower", aspect="auto", cmap="inferno")
    fig.colorbar(im, ax=ax, label="log10(star density + 1)")
    ax.set(xlabel=xlab, ylabel=ylab, title=f"{name.title()} sky density")
    fig.tight_layout(); fig.savefig(FIG / f"sky_density_{name}.png"); plt.show()

## Stellar radius, colour indices, and correlations
The original combined feature representation contains six corrected photometric inputs and eight inferred parameters. These figures preserve the feature definitions used for training.

In [ ]:
subset = balanced[(balanced["Teff"] > 0) & (balanced["Rad"] > 0)]
hist, xe, ye = np.histogram2d(subset["Teff"], np.log10(subset["Rad"]), bins=200)
fig, ax = plt.subplots(figsize=(8, 4))
im = ax.imshow(np.log10(hist.T + 1), extent=[xe[0], xe[-1], ye[0], ye[-1]], origin="lower", aspect="auto", cmap="inferno")
ax.set(xlabel="Effective temperature (K)", ylabel="log10(radius / solar radius)", title="Stellar radius vs temperature")
fig.colorbar(im, ax=ax, label="log10(star density + 1)"); fig.tight_layout()
fig.savefig(FIG / "radius_temperature.png"); plt.show()
for xcol, ycol in [("BP_RP_abs", "BP_G_abs"), ("BP_RP_abs", "G_RP_abs")]:
    fig, ax = plt.subplots(figsize=(6, 5))
    for cls, color in COLORS.items():
        subset = balanced[balanced["SpType-ELS"] == cls]
        ax.scatter(subset[xcol], subset[ycol], s=1, alpha=0.5, color=color, label=cls)
    ax.set(xlabel=xcol, ylabel=ycol, title="Corrected colour–colour diagram"); ax.legend(markerscale=4)
    fig.tight_layout(); fig.savefig(FIG / f"colors_{xcol}_{ycol}.png"); plt.show()
fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(balanced[FEATURE_SETS["combined_features"]].corr(), annot=True, fmt=".2f", cmap="magma", square=True, ax=ax)
ax.set_title("Corrected photometric and inferred parameter correlations")
fig.tight_layout(); fig.savefig(FIG / "feature_correlations.png"); plt.show()

## t-SNE (optional)
The original notebook used a 25,000-source random subset and standardized the combined features. Median imputation here avoids failures if non-FLAME quantities contain missing values. This visualization is descriptive only.

In [ ]:
if cfg["run_tsne"]:
    subset = balanced.sample(n=min(len(balanced), cfg["tsne_sample"]), random_state=cfg["random_state"])
    X = StandardScaler().fit_transform(SimpleImputer(strategy="median").fit_transform(subset[FEATURE_SETS["combined_features"]]))
    kwargs = {"n_components": 2, "random_state": cfg["random_state"], "init": "pca", "learning_rate": "auto", "perplexity": 30}
    kwargs["max_iter" if "max_iter" in inspect.signature(TSNE).parameters else "n_iter"] = 1000
    xy = TSNE(**kwargs).fit_transform(X)
    fig, ax = plt.subplots(figsize=(8, 6))
    for cls, color in COLORS.items():
        mask = (subset["SpType-ELS"] == cls).to_numpy()
        ax.scatter(xy[mask, 0], xy[mask, 1], s=4, alpha=0.6, color=color, label=cls)
    ax.set(xlabel="t-SNE 1", ylabel="t-SNE 2", title="Combined feature representation")
    ax.legend(markerscale=3); fig.tight_layout(); fig.savefig(FIG / "tsne_combined.png"); plt.show()
else:
    print("t-SNE disabled in config/experiment.json")